# Lab 03 — Color Spaces & Histograms

This notebook follows the exact flow of Lecture 03.
Work through each section in order — read the explanation, run the code, observe the output.

**Images used throughout (same as the lecture slides):**
- 🐒 **Mandrill** — rich colour, ideal for channel & HSV demos
- 👩 **Lenna** — classic CV test image, ideal for histogram & equalization demos
- 🍊 **Fruit tray** — real-world colour detection target

## How to work in this lab

1. Read the short explanation above each code cell.
2. Run the cell (`Shift+Enter`).
3. Look at the output carefully — compare it to the lecture slide.
4. Try changing one or two values and re-run.
5. Write one sentence: **what changed and why?**

## Setup — imports and display helper

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import urllib.request, os

def show(imgs, titles=None, cmap=None, cols=None):
    """Display one or several images side by side."""
    if not isinstance(imgs, (list, tuple)):
        imgs, titles = [imgs], [titles or ""]
    if titles is None:
        titles = [""] * len(imgs)
    n = len(imgs)
    cols = cols or n
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = np.array(axes).flatten()
    for ax, img, title in zip(axes, imgs, titles):
        if len(img.shape) == 2:
            ax.imshow(img, cmap=cmap or 'gray')
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=12)
        ax.axis('off')
    for ax in axes[n:]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

print("✅ Libraries loaded.")

## Load the three test images

The cell below looks for images in the course `test_images/` folder.
If you're running in Google Colab without the course folder, it downloads them automatically.

In [ ]:
# ── Image sources ─────────────────────────────────────────────────────────────
SOURCES = {
    "mandrill.jpg": [
        "https://homepages.cae.wisc.edu/~ece533/images/baboon.png",
        "https://upload.wikimedia.org/wikipedia/commons/a/a5/Mandrill_at_Bristol_Zoo.jpg",
    ],
    "lenna.png": [
        "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/lena.jpg",
        "https://homepages.cae.wisc.edu/~ece533/images/lena.bmp",
    ],
    "fruit_tray.gif": [],  # GIF — loaded via PIL below; no reliable public URL
}

# Local paths to check first (works when running from the course folder)
LOCAL = {
    "mandrill.jpg": ["../../../test_images/Mandrill.jpeg",
                     "test_images/Mandrill.jpeg", "Mandrill.jpeg"],
    "lenna.png":    ["../../../test_images/Lenna_(test_image) (1).png",
                     "test_images/Lenna_(test_image) (1).png"],
    "fruit_tray.gif": ["../../../test_images/fruit_tray.gif",
                       "test_images/fruit_tray.gif"],
}

def load_img(key):
    # 1. Try local paths
    for p in LOCAL[key]:
        if key.endswith(".gif"):
            if os.path.exists(p):
                frame = np.array(Image.open(p).convert("RGB"))
                return cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        else:
            img = cv2.imread(p)
            if img is not None:
                return img
    # 2. Try download
    for url in SOURCES.get(key, []):
        try:
            urllib.request.urlretrieve(url, key)
            img = cv2.imread(key)
            if img is not None:
                print(f"  Downloaded {key}")
                return img
        except Exception:
            continue
    return None

print("Loading images...")
mandrill = load_img("mandrill.jpg")
lenna    = load_img("lenna.png")
fruit    = load_img("fruit_tray.gif")

# Fallbacks if downloads fail
if mandrill is None:
    print("  ⚠️  Mandrill not found — using synthetic substitute")
    mandrill = np.zeros((256, 256, 3), dtype=np.uint8)
    mandrill[60:120, 60:130] = (0, 0, 200)    # red face patch (BGR)
    mandrill[60:100, 80:130] = (180, 30, 30)  # blue cheeks
    mandrill = cv2.GaussianBlur(mandrill, (15, 15), 0)

if lenna is None:
    print("  ⚠️  Lenna not found — using synthetic substitute")
    lenna = np.random.randint(80, 200, (256, 256, 3), dtype=np.uint8)
    lenna = cv2.GaussianBlur(lenna, (21, 21), 0)

if fruit is None:
    print("  ⚠️  Fruit tray not found — using synthetic substitute")
    fruit = np.ones((300, 400, 3), dtype=np.uint8) * 40
    for cx, cy in [(80,100),(200,80),(310,110),(130,220),(260,200)]:
        cv2.circle(fruit, (cx, cy), 40, (0, 100, 230), -1)   # orange in BGR
        cv2.circle(fruit, (cx, cy), 30, (0, 130, 255), -1)

print(f"✅ Mandrill: {mandrill.shape}")
print(f"✅ Lenna:    {lenna.shape}")
print(f"✅ Fruit:    {fruit.shape}")
show([mandrill, lenna, fruit], ["Mandrill", "Lenna", "Fruit Tray"])

---
## Part 1 — RGB Channel Separation

### Concept
Every colour image is **three grayscale layers stacked together**.
- The **Red channel** tells you how red each pixel is.
- The **Green channel** tells you how green.
- The **Blue channel** tells you how blue.

`cv2.split()` separates them into three independent 2D arrays.

⚠️ **OpenCV loads images in BGR order**, not RGB.
So `b, g, r = cv2.split(img)` — blue comes first!

### What to observe
Which channel is brightest on the Mandrill's face? Why?

In [ ]:
b, g, r = cv2.split(mandrill)

# Show each channel as grayscale (bright = high intensity of that colour)
show(
    [r, g, b],
    ["Red channel", "Green channel", "Blue channel"]
)

# Rebuild a red-only colour image to see it in colour
r_colour = np.zeros_like(mandrill)
r_colour[:, :, 2] = r   # place red values in the R slot (index 2 in BGR)
show([mandrill, r_colour], ["Original", "Red channel (in colour)"])

print(f"Each channel is a 2D array: shape = {r.shape}, dtype = {r.dtype}")

---
## Part 2 — Grayscale Histogram

### Concept
A histogram answers: **how many pixels have each intensity value (0–255)?**

- **X-axis**: pixel intensity (0 = black, 255 = white)
- **Y-axis**: how many pixels have that value

Reading the histogram before processing is like reading a map before navigating.
It tells you whether an image is over-exposed, under-exposed, or well-balanced.

### What to observe
Is Lenna dark or bright? Where is the bulk of the histogram?

In [ ]:
gray = cv2.cvtColor(lenna, cv2.COLOR_BGR2GRAY)
hist = cv2.calcHist([gray], [0], None, [256], [0, 256])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(gray, cmap='gray')
axes[0].set_title("Lenna — Grayscale")
axes[0].axis('off')

axes[1].plot(hist, color='black', linewidth=1.5)
axes[1].set_xlabel("Pixel intensity (0–255)")
axes[1].set_ylabel("Number of pixels")
axes[1].set_title("Grayscale histogram")
axes[1].set_xlim([0, 256])
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

peak = int(np.argmax(hist))
print(f"Most common intensity: {peak}  ({'bright' if peak > 127 else 'dark'} image)")
print(f"Total pixels: {int(np.sum(hist)):,}")

---
## Part 3 — Colour Histograms (per channel)

### Concept
Run a separate histogram for each BGR channel.
Three curves on one plot reveal the **colour signature** of an image:
- Red dominates → warm image (skin tones, sunset)
- Blue dominates → cool image (sky, water)
- All three overlap → neutral / grey

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].imshow(cv2.cvtColor(mandrill, cv2.COLOR_BGR2RGB))
axes[0].set_title("Mandrill")
axes[0].axis('off')

colors = ('b', 'g', 'r')
labels = ('Blue channel', 'Green channel', 'Red channel')
for i, (col, label) in enumerate(zip(colors, labels)):
    hist = cv2.calcHist([mandrill], [i], None, [256], [0, 256])
    axes[1].plot(hist, color=col, linewidth=1.5, label=label)

axes[1].set_xlabel("Pixel intensity (0–255)")
axes[1].set_ylabel("Number of pixels")
axes[1].set_title("BGR colour histograms — Mandrill")
axes[1].legend()
axes[1].set_xlim([0, 256])
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part 4 — HSV Colour Space

### Concept
HSV separates colour into three independent components:

| Channel | Meaning | Range (OpenCV) |
|---------|---------|----------------|
| **H** — Hue | The colour itself (red, green, blue…) | 0–180 |
| **S** — Saturation | How vivid the colour is (0 = grey) | 0–255 |
| **V** — Value | Brightness (0 = black) | 0–255 |

**Why is this better than RGB for detection?**
When lighting changes, only V changes. H and S stay stable.
That is the whole reason we use HSV.

⚠️ OpenCV Hue range is **0–180**, not 0–360. Always halve standard Hue tables.

In [ ]:
hsv = cv2.cvtColor(mandrill, cv2.COLOR_BGR2HSV)
h, s, v = cv2.split(hsv)

show(
    [mandrill, h, s, v],
    ["Original", "H — Hue", "S — Saturation", "V — Value"],
    cols=4
)

# Show Hue as a coloured wheel for intuition
show([h], ["Hue channel (colour-mapped)"], cmap='hsv')

# Print some pixel values to build intuition
print("Sample pixel at centre:")
cy, cx = mandrill.shape[0]//2, mandrill.shape[1]//2
print(f"  BGR: {mandrill[cy, cx]}")
print(f"  HSV: {hsv[cy, cx]}  → H={hsv[cy,cx,0]}, S={hsv[cy,cx,1]}, V={hsv[cy,cx,2]}")

---
## Part 5 — Colour Thresholding with `cv2.inRange()`

### Concept
`cv2.inRange(hsv, lower, upper)` creates a **binary mask**:
- **255 (white)** wherever ALL three channels fall inside the range
- **0 (black)** everywhere else

Then `cv2.bitwise_and(img, img, mask=mask)` extracts only the matching pixels.

**Strategy for robust thresholds:**
- H → narrow (±10–15 from target hue)
- S → from ~100 (exclude grey/washed-out pixels)
- V → wide (50–255) to cover shadows and highlights

In [ ]:
hsv_fruit = cv2.cvtColor(fruit, cv2.COLOR_BGR2HSV)

# Orange: H ≈ 8–25 in OpenCV (0–180 scale)
lower_orange = np.array([8,  100, 80])
upper_orange = np.array([25, 255, 255])

mask = cv2.inRange(hsv_fruit, lower_orange, upper_orange)

# Optional cleanup: close small holes in the mask
kernel = np.ones((7, 7), np.uint8)
mask_clean = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

result = cv2.bitwise_and(fruit, fruit, mask=mask_clean)

show(
    [fruit, mask, mask_clean, result],
    ["Original", "Raw mask", "Cleaned mask", "Detected oranges"],
    cols=4
)

orange_pixels = int(np.sum(mask_clean > 0))
total_pixels  = mask_clean.size
print(f"Orange pixels: {orange_pixels:,} ({100*orange_pixels/total_pixels:.1f}% of image)")

---
## Part 6 — Detecting Red: The Wrap-Around Problem

### Concept
Red is **the only colour that wraps around the Hue axis** in HSV.
It lives near H=0 **and** near H=180 simultaneously.

A single `inRange()` call misses half the red pixels.
**Solution: two ranges + `cv2.bitwise_or()`.**

In [ ]:
hsv_m = cv2.cvtColor(mandrill, cv2.COLOR_BGR2HSV)

# Red wraps: needs two ranges
lower1, upper1 = np.array([0,  80, 50]), np.array([10, 255, 255])
lower2, upper2 = np.array([165, 80, 50]), np.array([180, 255, 255])

mask1 = cv2.inRange(hsv_m, lower1, upper1)
mask2 = cv2.inRange(hsv_m, lower2, upper2)
mask_red = cv2.bitwise_or(mask1, mask2)   # combine both ranges

result_red = cv2.bitwise_and(mandrill, mandrill, mask=mask_red)

show(
    [mandrill, mask1, mask2, mask_red, result_red],
    ["Original", "Range 1 (H 0–10)", "Range 2 (H 165–180)",
     "Combined mask", "Red regions only"],
    cols=5
)

---
## Part 7 — Why HSV is Robust to Lighting

### Concept
A red tomato under **bright** light: BGR ≈ (30, 30, 230)
The same tomato under **dim** light: BGR ≈ (10, 10, 80)

In RGB these look completely different — a single threshold fails.
In HSV: both have H ≈ 0, S ≈ 230+. Only V changes.

**Rule of thumb:** narrow H, medium S, wide V.

In [ ]:
# Simulate: same orange block under three lighting conditions
def orange_patch(brightness):
    """Create an orange patch at a given brightness level."""
    patch = np.zeros((100, 100, 3), dtype=np.uint8)
    patch[:] = (0, int(165 * brightness), int(255 * brightness))  # BGR orange
    return patch

bright = orange_patch(1.0)
medium = orange_patch(0.5)
dim    = orange_patch(0.2)

# Print BGR and HSV values for each
for name, img in [("Bright", bright), ("Medium", medium), ("Dim", dim)]:
    bgr = img[50, 50]
    hsv_val = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)[50, 50]
    print(f"{name:6s}  BGR={bgr}   HSV=(H={hsv_val[0]}, S={hsv_val[1]}, V={hsv_val[2]})")

# Detect orange with WIDE V bounds in HSV
canvas = np.hstack([bright, medium, dim])
hsv_canvas = cv2.cvtColor(canvas, cv2.COLOR_BGR2HSV)

lower = np.array([8,  60, 30])   # very wide V → catches even dim light
upper = np.array([25, 255, 255])
mask_hsv = cv2.inRange(hsv_canvas, lower, upper)
result_hsv = cv2.bitwise_and(canvas, canvas, mask=mask_hsv)

show(
    [canvas, mask_hsv, result_hsv],
    ["Original (bright / medium / dim)", "HSV mask (wide V)", "Detected in all conditions"]
)

---
## Part 8 — Histogram Equalization & CLAHE

### Concept
Histogram equalization **redistributes** pixel intensities so they span 0–255 evenly.
Useful for low-contrast or poorly-lit images.

Two methods:
| Method | When to use |
|--------|-------------|
| `cv2.equalizeHist()` | Global — works well on uniformly dark images |
| `cv2.createCLAHE()` | Local (tile-based) — better for images with uneven lighting |

⚠️ **For colour images: equalize only the V channel in HSV.**
Equalizing BGR directly distorts the colours.

In [ ]:
# Darken Lenna to simulate poor lighting
lenna_gray = cv2.cvtColor(lenna, cv2.COLOR_BGR2GRAY)
lenna_dark = cv2.convertScaleAbs(lenna_gray, alpha=0.4, beta=0)

# Global equalization
eq_global = cv2.equalizeHist(lenna_dark)

# CLAHE (Contrast Limited Adaptive Histogram Equalization)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
eq_clahe = clahe.apply(lenna_dark)

show(
    [lenna_dark, eq_global, eq_clahe],
    ["Dark original", "Global equalizeHist()", "CLAHE (clipLimit=2.0)"]
)

# Compare histograms side-by-side
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for ax, img, title in zip(axes,
                           [lenna_dark, eq_global, eq_clahe],
                           ["Dark", "Global EQ", "CLAHE"]):
    h = cv2.calcHist([img], [0], None, [256], [0, 256])
    ax.plot(h, color='steelblue', linewidth=1)
    ax.set_title(f"Histogram — {title}")
    ax.set_xlim([0, 256])
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Notice: CLAHE avoids over-amplifying already-bright areas (fewer extreme spikes).")

In [ ]:
# Equalize a COLOUR image correctly — via V channel in HSV
lenna_hsv = cv2.cvtColor(lenna, cv2.COLOR_BGR2HSV)
lenna_hsv_dark = lenna_hsv.copy()
lenna_hsv_dark[:, :, 2] = cv2.convertScaleAbs(lenna_hsv[:, :, 2], alpha=0.4)

dark_colour = cv2.cvtColor(lenna_hsv_dark, cv2.COLOR_HSV2BGR)

# Equalize only V
lenna_hsv_eq = lenna_hsv_dark.copy()
lenna_hsv_eq[:, :, 2] = clahe.apply(lenna_hsv_dark[:, :, 2])
eq_colour = cv2.cvtColor(lenna_hsv_eq, cv2.COLOR_HSV2BGR)

show(
    [lenna, dark_colour, eq_colour],
    ["Original Lenna", "Darkened colour", "CLAHE on V channel only"]
)
print("Colours are preserved — only brightness was adjusted.")

---
## Summary — what you built in this lab

| Concept | Key function | When to use |
|---------|-------------|-------------|
| Channel separation | `cv2.split(img)` | Understand which colour carries signal |
| Histogram | `cv2.calcHist()` | Diagnose image quality before processing |
| HSV conversion | `cv2.cvtColor(BGR2HSV)` | Prepare for robust colour detection |
| Colour threshold | `cv2.inRange(hsv, lo, hi)` | Segment objects by colour |
| Red detection | Two ranges + `bitwise_or` | Always needed for red (wrap-around at H=0/180) |
| Equalization | `equalizeHist()` / `CLAHE` | Improve contrast in dark images |

**Next:** complete the Independent Work notebook — detect and count orange fruits using contours.